# Thalamus — fish species classifier (Colab)

Colab-friendly wrapper around `scripts/fish_classifier_train.py`. The training
logic lives in the repo, not in this notebook, so a run here and a run on a
workstation stay identical.

**Runtime → Change runtime type → GPU** before you start.

Steps: install deps → get data → set config → train → export → download.

## 1. Check the GPU

If this reports no GPU, training will still run but will be roughly an order of
magnitude slower.

In [ ]:
!nvidia-smi || echo 'No GPU allocated — check Runtime > Change runtime type'

## 2. Clone the repo and install dependencies

Colab ships with torch already installed, so `torch` is skipped here to avoid a
long reinstall that usually resolves to the same version.

In [ ]:
import os

REPO_URL = 'https://github.com/conradpower279/Thalamus.git'
REPO_DIR = '/content/Thalamus'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull

In [ ]:
# timm/albumentations/onnx pinned to match requirements.txt; torch comes from Colab.
!pip install -q timm==1.0.11 albumentations==1.4.21 onnx==1.17.0 PyYAML==6.0.2

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. Get the data

Colab's local disk is wiped when the runtime recycles, so keep the dataset in
Drive and copy it in. The archive should unpack to the layout described in
`data/README.md`:

```
data/train/<species>/*.jpg
data/val/<species>/*.jpg
```

Copying a zip and unpacking to local disk is much faster than reading thousands
of small files directly off a mounted Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust to wherever your dataset archive lives in Drive.
ARCHIVE = '/content/drive/MyDrive/thalamus/fish_dataset.zip'

!mkdir -p /content/Thalamus/data
!unzip -q -o {ARCHIVE} -d /content/Thalamus/data
!ls /content/Thalamus/data

### Optional: build the splits

Only if your archive is one folder per species with no train/val/test split yet.

In [ ]:
# !python scripts/split_data.py --source /content/Thalamus/raw --output /content/Thalamus/data

### Check class balance

Worth a glance before every run. The training script uses unweighted
cross-entropy, so a heavily skewed dataset produces a model that quietly ignores
the rare species while still reporting good accuracy.

In [ ]:
from pathlib import Path

for split in ['train', 'val', 'test']:
    split_dir = Path('data') / split
    if not split_dir.is_dir():
        continue
    counts = {
        d.name: len(list(d.glob('*.jpg'))) + len(list(d.glob('*.png')))
        for d in sorted(split_dir.iterdir()) if d.is_dir()
    }
    print(f'{split}: {sum(counts.values())} images across {len(counts)} species')
    for name, count in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f'   {count:>6}  {name}')
    print()

## 4. Configure

Defaults come from `configs/default.yaml`. Review them here, then either edit
that file or pass overrides on the command line in the next cell.

In [ ]:
import yaml

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

for key, value in config.items():
    print(f'{key:>20}: {value}')

## 5. Train

Stage 1 trains the classifier head on the frozen backbone; stage 2 unfreezes
everything and fine-tunes with a cosine schedule. Checkpoints and the ONNX
export are written to `models/`.

Colab disconnects idle sessions — for long runs, keep the tab visible.

In [ ]:
!python scripts/fish_classifier_train.py --config configs/default.yaml --data-dir data

## 6. Back up the checkpoints

Do this before the runtime recycles, or the run is lost.

In [ ]:
!mkdir -p /content/drive/MyDrive/thalamus/models
!cp -v models/*.pth models/*.onnx models/classes.json /content/drive/MyDrive/thalamus/models/

## 7. Export to TFLite for the Flutter app

The conversion toolchain is large and conflicts with the training environment's
pins, so it is installed separately. Expect to restart the runtime afterwards.

For an int8 model, pass `--calib-dir data/val` — calibration needs real images.

In [ ]:
!pip install -q -r requirements-export.txt

In [ ]:
!python export/export_tflite.py --quantization float16
!ls -lh export/output/

## 8. Download the app artifacts

`classes.json` is not optional — the TFLite graph emits bare indices, and
without the mapping the app cannot name a species. Always keep the two together.

In [ ]:
from google.colab import files
from pathlib import Path

for path in sorted(Path('export/output').glob('*.tflite')):
    files.download(str(path))

files.download('export/output/classes.json')